# Practice 5. Dynamic Web Scraping & Data Cleaning

**Student:** Tomiris (ID 23B031507) · KBTU, Data Storage and Analysis

**Source website:** https://quotes.toscrape.com  (a public scraping sandbox, no login required)

* Section A: `https://quotes.toscrape.com/js/` - quotes are rendered by JavaScript, next pages are reached by clicking **Next**
* Section B: `https://quotes.toscrape.com/scroll` - quotes are loaded in batches while **scrolling** (infinite scroll)
* Section C: author pages (`/author/...`) opened for each unique author (born date, location, biography)

**Setup (run once in the terminal):**
```
pip install playwright pandas pandera python-docx jupyter
playwright install chromium
```

In [1]:
import re, time
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 160)

In [2]:
import sys
import asyncio

print(sys.executable)
print(sys.version)
print(asyncio.get_event_loop_policy())

C:\Users\Владелец\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe
3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


## Task 1. Dynamic data collection (Playwright)

In [3]:

import asyncio
import re
import pandas as pd

from urllib.parse import urljoin
from playwright.async_api import async_playwright


BASE_URL = "https://quotes.toscrape.com"
OUTPUT_FILE = "raw_dataset.csv"


def make_author_url(author_name):
    slug = re.sub(r"[^A-Za-z0-9]+", "-", author_name).strip("-")
    return urljoin(BASE_URL, f"/author/{slug}/")


async def extract_quotes(page, source, page_no):
    results = []
    cards = page.locator(".quote")

    for i in range(await cards.count()):
        card = cards.nth(i)

        try:
            quote_text = (await card.locator(".text").inner_text()).strip()
            author = (await card.locator(".author").inner_text()).strip()
            tags = await card.locator(".tags .tag").all_inner_texts()

            results.append({
                "text": quote_text,
                "author": author,
                "tags": ", ".join(tag.strip() for tag in tags),
                "source": source,
                "page_no": page_no,
                "author_url": make_author_url(author)
            })

        except Exception as error:
            print(f"Skipping quote: {error}")

    return results


async def scrape_js_pagination(page):
    print("Scraping JavaScript pagination...")

    await page.goto(f"{BASE_URL}/js/", wait_until="domcontentloaded")
    await page.locator(".quote").first.wait_for()

    results = []
    seen = set()
    page_no = 1

    while True:
        quotes = await extract_quotes(page, "js_pagination", page_no)
        new_count = 0

        for quote in quotes:
            if quote["text"] not in seen:
                seen.add(quote["text"])
                results.append(quote)
                new_count += 1

        print(f"JS page {page_no}: {new_count} new quotes")

        next_button = page.locator("li.next a")
        if await next_button.count() == 0:
            break

        old_text = await page.locator(".quote .text").first.inner_text()
        await next_button.click()

        try:
            await page.wait_for_function(
                """oldText => {
                    const el = document.querySelector(".quote .text");
                    return el && el.textContent.trim() !== oldText.trim();
                }""",
                arg=old_text,
                timeout=10000
            )
        except Exception:
            print("Pagination stopped: page content did not change.")
            break

        page_no += 1

    print(f"JS pagination total: {len(results)}")
    return results


async def scrape_infinite_scroll(page):
    print("\nScraping infinite scroll...")

    await page.goto(f"{BASE_URL}/scroll", wait_until="domcontentloaded")
    await page.locator(".quote").first.wait_for()

    results = []
    seen = set()
    unchanged_rounds = 0

    for round_no in range(1, 31):
        quotes = await extract_quotes(page, "infinite_scroll", round_no)
        new_count = 0

        for quote in quotes:
            if quote["text"] not in seen:
                seen.add(quote["text"])
                results.append(quote)
                new_count += 1

        print(f"Scroll round {round_no}: {new_count} new quotes")

        if new_count == 0:
            unchanged_rounds += 1
        else:
            unchanged_rounds = 0

        if unchanged_rounds >= 3:
            break

        old_count = await page.locator(".quote").count()

        await page.evaluate(
            "window.scrollTo(0, document.body.scrollHeight)"
        )

        try:
            await page.wait_for_function(
                """oldCount =>
                    document.querySelectorAll(".quote").length > oldCount
                """,
                arg=old_count,
                timeout=5000
            )
        except Exception:
            await page.wait_for_timeout(1000)

    print(f"Infinite scroll total: {len(results)}")
    return results


async def scrape_author_details(page, author_url):
    details = {
        "born_date": None,
        "born_location": None,
        "bio": None
    }

    try:
        response = await page.goto(
            author_url,
            wait_until="domcontentloaded",
            timeout=15000
        )

        if response is None or response.status >= 400:
            print(f"Could not open author page: {author_url}")
            return details

        selectors = {
            "born_date": ".author-born-date",
            "born_location": ".author-born-location",
            "bio": ".author-description"
        }

        for field, selector in selectors.items():
            try:
                value = await page.locator(selector).inner_text(timeout=3000)
                details[field] = value.strip() or None
            except Exception:
                pass

    except Exception as error:
        print(f"Author page error: {error}")

    return details


async def run_scraper():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)

        try:
            page = await browser.new_page()
            page.set_default_timeout(10000)

            js_quotes = await scrape_js_pagination(page)
            scroll_quotes = await scrape_infinite_scroll(page)

            all_quotes = js_quotes + scroll_quotes

            author_urls = sorted({
                quote["author_url"]
                for quote in all_quotes
                if quote.get("author_url")
            })

            print(f"\nUnique author URLs: {len(author_urls)}")

            author_details = {}

            for index, author_url in enumerate(author_urls, start=1):
                print(f"Scraping author {index}/{len(author_urls)}")

                author_details[author_url] = await scrape_author_details(
                    page, author_url
                )

            for quote in all_quotes:
                quote.update(
                    author_details.get(
                        quote["author_url"],
                        {
                            "born_date": None,
                            "born_location": None,
                            "bio": None
                        }
                    )
                )

            return all_quotes

        finally:
            await browser.close()


async def main():
    quotes = await run_scraper()

    if not quotes:
        print("No quotes collected. Check the website connection.")
        return

    columns = [
        "text",
        "author",
        "tags",
        "source",
        "page_no",
        "author_url",
        "born_date",
        "born_location",
        "bio"
    ]

    df = pd.DataFrame(quotes)

    for column in columns:
        if column not in df.columns:
            df[column] = None

    df = df[columns]
    df["scraped_at"] = pd.Timestamp.now().floor("s").isoformat()

    df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print("\n========== RESULTS ==========")
    print(f"Total rows: {len(df)}")
    print("JS pagination:", (df["source"] == "js_pagination").sum())
    print("Infinite scroll:", (df["source"] == "infinite_scroll").sum())
    print("Unique authors:", df["author"].nunique())
    print("Dataset shape:", df.shape)
    print("Saved file:", OUTPUT_FILE)

    print("\nMissing values:")
    print(df.isnull().sum())

    print("\nPreview:")
    print(df.head().to_string(index=False))

    return df


# Run Playwright in a separate thread with a Windows Proactor event loop.
def run_with_proactor():
    loop = asyncio.ProactorEventLoop()
    asyncio.set_event_loop(loop)

    try:
        return loop.run_until_complete(main())
    finally:
        loop.close()
        asyncio.set_event_loop(None)


df = await asyncio.to_thread(run_with_proactor)


Scraping JavaScript pagination...
JS page 1: 10 new quotes
JS page 2: 10 new quotes
JS page 3: 10 new quotes
JS page 4: 10 new quotes
JS page 5: 10 new quotes
JS page 6: 10 new quotes
JS page 7: 10 new quotes
JS page 8: 10 new quotes
JS page 9: 10 new quotes
JS page 10: 10 new quotes
JS pagination total: 100

Scraping infinite scroll...
Scroll round 1: 10 new quotes
Scroll round 2: 10 new quotes
Scroll round 3: 10 new quotes
Scroll round 4: 10 new quotes
Scroll round 5: 10 new quotes
Scroll round 6: 10 new quotes
Scroll round 7: 10 new quotes
Scroll round 8: 10 new quotes
Scroll round 9: 10 new quotes
Scroll round 10: 10 new quotes
Scroll round 11: 0 new quotes
Scroll round 12: 0 new quotes
Scroll round 13: 0 new quotes
Infinite scroll total: 100

Unique author URLs: 50
Scraping author 1/50
Scraping author 2/50
Scraping author 3/50
Scraping author 4/50
Scraping author 5/50
Scraping author 6/50
Scraping author 7/50
Scraping author 8/50
Scraping author 9/50
Scraping author 10/50
Scraping

## Task 2. Missing values
Empty strings and label-only values are not seen by `isna()`, so they are first converted to `NaN`.

In [4]:

import pandas as pd
import numpy as np
from pathlib import Path

# 0) Load the raw dataset created by the scraper
file_path = Path("raw_dataset.csv")

if not file_path.exists():
    raise FileNotFoundError(
        "raw_dataset.csv was not found. "
        "Check that the CSV file is in the current Jupyter folder."
    )

df_raw = pd.read_csv(file_path)

# Map scraper column names to the names expected by this cleaning step
if "tags_raw" not in df_raw.columns and "tags" in df_raw.columns:
    df_raw = df_raw.rename(columns={"tags": "tags_raw"})

# 1) Missing values exactly as scraped
df = df_raw.copy()
missing_raw = df_raw.isna().sum()

# 2) Hidden missing values: empty / whitespace strings
# and the label "Tags:" without any tag
df = df.replace(r"^\s*$", np.nan, regex=True)

if "tags_raw" not in df.columns:
    raise KeyError(
        "The dataset has no 'tags_raw' or 'tags' column. "
        "Check the CSV column names."
    )

label_only = df["tags_raw"].astype("string").str.fullmatch(
    r"\s*Tags:\s*", na=False
)

n_label_only_tags = int(label_only.sum())
df.loc[label_only, "tags_raw"] = np.nan

missing_marked = df.isna().sum()

# 3) Handling missing values
# Text and author are essential
essential = df[["text", "author"]].isna().any(axis=1)

n_dropped_missing = int(essential.sum())
df = df.loc[~essential].copy()

# Missing tags get an explicit label
df["tags_raw"] = df["tags_raw"].fillna("Tags: untagged")

# Author details remain missing when unavailable
missing_after = df.isna().sum()

missing_table = pd.DataFrame({
    "as scraped": missing_raw,
    "after marking hidden NaN": missing_marked,
    "after processing": missing_after
}).fillna(0).astype(int)

print("Rows loaded:", len(df_raw))
print("Rows dropped (missing text/author):", n_dropped_missing)
print("Quotes with 'Tags:' label but no tags:", n_label_only_tags)
print("Rows after processing:", len(df))

display(missing_table)
display(df.head())


Rows loaded: 200
Rows dropped (missing text/author): 0
Quotes with 'Tags:' label but no tags: 0
Rows after processing: 200


,as scraped,after marking hidden NaN,after processing
text,0,0,0
author,0,0,0
tags_raw,6,6,0
source,0,0,0
page_no,0,0,0
author_url,0,0,0
born_date,4,4,4
born_location,4,4,4
bio,4,4,4
scraped_at,0,0,0


,text,author,tags_raw,source,page_no,author_url,born_date,born_location,bio,scraped_at
0,“The world as we have created it is a process of our thinking. It cannot be ...,Albert Einstein,"change, deep-thoughts, thinking, world",js_pagination,1,https://quotes.toscrape.com/author/Albert-Einstein/,"March 14, 1879","in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germany. He completed his Ph.D. at...",2026-11-06T09:06:21
1,"“It is our choices, Harry, that show what we truly are, far more than our ab...",J.K. Rowling,"abilities, choices",js_pagination,1,https://quotes.toscrape.com/author/J-K-Rowling/,"July 31, 1965","in Yate, South Gloucestershire, England, The United Kingdom",See also: Robert GalbraithAlthough she writes under the pen name J.K. Rowlin...,2026-11-06T09:06:21
2,“There are only two ways to live your life. One is as though nothing is a mi...,Albert Einstein,"inspirational, life, live, miracle, miracles",js_pagination,1,https://quotes.toscrape.com/author/Albert-Einstein/,"March 14, 1879","in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germany. He completed his Ph.D. at...",2026-11-06T09:06:21
3,"“The person, be it gentleman or lady, who has not pleasure in a good novel, ...",Jane Austen,"aliteracy, books, classic, humor",js_pagination,1,https://quotes.toscrape.com/author/Jane-Austen/,"December 16, 1775","in Steventon Rectory, Hampshire, The United Kingdom","Jane Austen was an English novelist whose works of romantic fiction, set amo...",2026-11-06T09:06:21
4,"“Imperfection is beauty, madness is genius and it's better to be absolutely ...",Marilyn Monroe,"be-yourself, inspirational",js_pagination,1,https://quotes.toscrape.com/author/Marilyn-Monroe/,"June 01, 1926",in The United States,"Marilyn Monroe (born Norma Jeane Mortenson; June 1, 1926 – August 5, 1962) w...",2026-11-06T09:06:21


## Task 3. String cleaning (pandas `.str` + regex)

In [5]:

import pandas as pd
import numpy as np
import re

# Load the raw dataset
df_raw = pd.read_csv("raw_dataset.csv")
df = df_raw.copy()

# Ensure the column names match the cleaning pipeline
if "tags_raw" not in df.columns and "tags" in df.columns:
    df.rename(columns={"tags": "tags_raw"}, inplace=True)

# Add expected columns if they are absent
for col in ["text", "author", "author_url", "tags_raw"]:
    if col not in df.columns:
        df[col] = np.nan

# Normalize text and author
df["text"] = df["text"].astype("string").str.replace(
    r"\s+", " ", regex=True
).str.strip()

df["author"] = df["author"].astype("string").str.replace(
    r"\s+", " ", regex=True
).str.strip()

# Extract author slug
df["author_slug"] = df["author_url"].astype("string").str.extract(
    r"/author/([^/]+)", expand=False
)

# Mark empty strings as missing
df = df.replace(r"^\s*$", np.nan, regex=True)

# Handle missing or empty tags
label_only = df["tags_raw"].astype("string").str.fullmatch(
    r"\s*Tags:\s*", na=False
)
df.loc[label_only, "tags_raw"] = np.nan

df["tags_raw"] = df["tags_raw"].fillna("Tags: untagged")

# Clean quote text
df["text_clean"] = (
    df["text"]
    .astype("string")
    .str.replace("“", "", regex=False)
    .str.replace("”", "", regex=False)
    .str.strip()
)

# Clean tags: lowercase, remove the "Tags:" prefix if present
df["tags_clean"] = (
    df["tags_raw"]
    .astype("string")
    .str.replace(r"^\s*Tags:\s*", "", regex=True)
    .str.lower()
    .str.replace(r"\s+", "", regex=True)
    .str.strip()
)

df.loc[
    df["tags_clean"].isin(["", "nan", "none"]),
    "tags_clean"
] = "untagged"

# Word and tag counts
df["word_count"] = df["text_clean"].fillna("").str.split().str.len()

df["tags_count"] = df["tags_clean"].apply(
    lambda value: 0 if value == "untagged"
    else len([tag for tag in value.split(",") if tag])
)

# Author details
if "born_date" in df.columns:
    df["born_date"] = pd.to_datetime(
        df["born_date"], errors="coerce"
    )
else:
    df["born_date"] = pd.NaT

df["born_year"] = df["born_date"].dt.year.astype("Int64")

if "born_location" in df.columns:
    locations = df["born_location"].astype("string")
else:
    locations = pd.Series(pd.NA, index=df.index, dtype="string")

df["born_city"] = locations.str.split(",").str[0].str.strip()
df["born_country"] = locations.str.split(",").str[-1].str.strip()

if "bio" in df.columns:
    df["bio_words"] = (
        df["bio"].fillna("").astype(str).str.split().str.len()
    ).astype("Int64")
else:
    df["bio_words"] = pd.Series(
        pd.NA, index=df.index, dtype="Int64"
    )

# Remove records without essential quote text or author
df_clean = df.dropna(subset=["text_clean", "author"]).copy()

print("Raw rows:", len(df_raw))
print("Clean rows:", len(df_clean))
print("Columns:", df_clean.columns.tolist())

display(df_clean.head())


Raw rows: 200
Clean rows: 200
Columns: ['text', 'author', 'tags_raw', 'source', 'page_no', 'author_url', 'born_date', 'born_location', 'bio', 'scraped_at', 'author_slug', 'text_clean', 'tags_clean', 'word_count', 'tags_count', 'born_year', 'born_city', 'born_country', 'bio_words']


,text,author,tags_raw,source,page_no,author_url,born_date,born_location,bio,scraped_at,author_slug,text_clean,tags_clean,word_count,tags_count,born_year,born_city,born_country,bio_words
0,“The world as we have created it is a process of our thinking. It cannot be ...,Albert Einstein,"change, deep-thoughts, thinking, world",js_pagination,1,https://quotes.toscrape.com/author/Albert-Einstein/,1879-03-14,"in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germany. He completed his Ph.D. at...",2026-11-06T09:06:21,Albert-Einstein,The world as we have created it is a process of our thinking. It cannot be c...,"change,deep-thoughts,thinking,world",21,4,1879,in Ulm,Germany,605
1,"“It is our choices, Harry, that show what we truly are, far more than our ab...",J.K. Rowling,"abilities, choices",js_pagination,1,https://quotes.toscrape.com/author/J-K-Rowling/,1965-07-31,"in Yate, South Gloucestershire, England, The United Kingdom",See also: Robert GalbraithAlthough she writes under the pen name J.K. Rowlin...,2026-11-06T09:06:21,J-K-Rowling,"It is our choices, Harry, that show what we truly are, far more than our abi...","abilities,choices",16,2,1965,in Yate,The United Kingdom,629
2,“There are only two ways to live your life. One is as though nothing is a mi...,Albert Einstein,"inspirational, life, live, miracle, miracles",js_pagination,1,https://quotes.toscrape.com/author/Albert-Einstein/,1879-03-14,"in Ulm, Germany","In 1879, Albert Einstein was born in Ulm, Germany. He completed his Ph.D. at...",2026-11-06T09:06:21,Albert-Einstein,There are only two ways to live your life. One is as though nothing is a mir...,"inspirational,life,live,miracle,miracles",26,5,1879,in Ulm,Germany,605
3,"“The person, be it gentleman or lady, who has not pleasure in a good novel, ...",Jane Austen,"aliteracy, books, classic, humor",js_pagination,1,https://quotes.toscrape.com/author/Jane-Austen/,1775-12-16,"in Steventon Rectory, Hampshire, The United Kingdom","Jane Austen was an English novelist whose works of romantic fiction, set amo...",2026-11-06T09:06:21,Jane-Austen,"The person, be it gentleman or lady, who has not pleasure in a good novel, m...","aliteracy,books,classic,humor",19,4,1775,in Steventon Rectory,The United Kingdom,313
4,"“Imperfection is beauty, madness is genius and it's better to be absolutely ...",Marilyn Monroe,"be-yourself, inspirational",js_pagination,1,https://quotes.toscrape.com/author/Marilyn-Monroe/,1926-06-01,in The United States,"Marilyn Monroe (born Norma Jeane Mortenson; June 1, 1926 – August 5, 1962) w...",2026-11-06T09:06:21,Marilyn-Monroe,"Imperfection is beauty, madness is genius and it's better to be absolutely r...","be-yourself,inspirational",16,2,1926,in The United States,in The United States,365


## Task 4. Data validation (Pandera)

In [6]:
import pandera.pandas as pa
from pandera.pandas import Column, Check, DataFrameSchema

schema = DataFrameSchema(
    {
        "source": Column(str, Check.isin(["js_pagination", "infinite_scroll"])),
        "page_no": Column(int, Check.in_range(1, 20)),
        "author": Column(str, Check.str_length(min_value=2)),
        "author_slug": Column(str, Check.str_matches(r"^[A-Za-z0-9\-]+$")),
        "text_clean": Column(str, Check.str_length(min_value=10, max_value=1000)),
        "word_count": Column(int, Check.in_range(2, 300)),
        "tags_clean": Column(str, Check.str_matches(r"^[a-z0-9\-,]+$")),
        "tags_count": Column(int, Check.in_range(0, 15)),
        "born_date": Column(pa.DateTime, Check.in_range(pd.Timestamp("1000-01-01"), pd.Timestamp("2026-12-31")), nullable=True),
        "born_year": Column("Int64", Check.in_range(1000, 2026), nullable=True),
        "born_city": Column(str, nullable=True),
        "born_country": Column(str, nullable=True),
        "bio_words": Column("Int64", Check.ge(0), nullable=True),
    },
    unique=["author", "text_clean"],
    coerce=True,
)

validation_errors_first = 0
n_rows_removed_validation = 0
try:
    schema.validate(df_clean, lazy=True)
    print("Validation passed on the first run")
except pa.errors.SchemaErrors as err:
    validation_errors_first = len(err.failure_cases)
    print("Problems found:", validation_errors_first)
    display(err.failure_cases.head(20))
    # correction: remove records that violate the rules, then validate again
    bad_idx = err.failure_cases["index"].dropna().astype(int).unique()
    n_rows_removed_validation = len(bad_idx)
    df_clean = df_clean.drop(index=bad_idx).reset_index(drop=True)

df_valid = schema.validate(df_clean, lazy=True)
print("Validation after correction: OK,", len(df_valid), "rows")

Problems found: 402


,schema_context,column,check,check_number,failure_case,index
0,DataFrameSchema,author,multiple_fields_uniqueness,None,Albert Einstein,0
264,DataFrameSchema,text_clean,multiple_fields_uniqueness,None,Not all those who wander are lost.,64
274,DataFrameSchema,text_clean,multiple_fields_uniqueness,None,"What really knocks me out is a book that, when you're all done reading it, y...",74
273,DataFrameSchema,text_clean,multiple_fields_uniqueness,None,"Think left and think right and think low and think high. Oh, the thinks you ...",73
272,DataFrameSchema,text_clean,multiple_fields_uniqueness,None,"The trouble with having an open mind, of course, is that people will insist ...",72
271,DataFrameSchema,text_clean,multiple_fields_uniqueness,None,Some people never go crazy. What truly horrible lives they must lead.,71
270,DataFrameSchema,text_clean,multiple_fields_uniqueness,None,If I had a flower for every time I thought of you...I could walk through my ...,70
269,DataFrameSchema,text_clean,multiple_fields_uniqueness,None,I have heard there are troubles of more than one kind. Some come from ahead ...,69
268,DataFrameSchema,text_clean,multiple_fields_uniqueness,None,I have never let my schooling interfere with my education.,68
267,DataFrameSchema,text_clean,multiple_fields_uniqueness,None,Finish each day and be done with it. You have done what you could. Some blun...,67


Validation after correction: OK, 0 rows


## Task 5. Final dataset

In [ ]:
df_final = df_valid.copy()
df_final.to_csv("final_dataset.csv", index=False)

print("Shape:", df_final.shape)
print("Duplicates:", int(df_final.duplicated().sum()))
print("\nMissing values:")
display(df_final.isna().sum().to_frame("missing"))
print("Data types:")
display(df_final.dtypes.astype(str).to_frame("dtype"))
display(df_final.head(10))

compare = pd.DataFrame({
    "original scraped": [df_raw.shape[0], df_raw.shape[1], int(df_raw.isna().sum().sum()), int(df_raw.duplicated(subset=["author", "text"]).sum())],
    "cleaned": [df_final.shape[0], df_final.shape[1], int(df_final.isna().sum().sum()), int(df_final.duplicated(subset=["author", "text_clean"]).sum())],
}, index=["rows", "columns", "missing cells", "duplicate quotes"])
display(compare)

Shape: (0, 19)
Duplicates: 0

Missing values:


,missing
text,0
author,0
tags_raw,0
source,0
page_no,0
author_url,0
born_date,0
born_location,0
bio,0
scraped_at,0


Data types:


,dtype
text,string
author,object
tags_raw,object
source,object
page_no,int64
author_url,object
born_date,datetime64[ns]
born_location,object
bio,object
scraped_at,object


,text,author,tags_raw,source,page_no,author_url,born_date,born_location,bio,scraped_at,author_slug,text_clean,tags_clean,word_count,tags_count,born_year,born_city,born_country,bio_words


,original scraped,cleaned
rows,200,0
columns,10,19
missing cells,18,0
duplicate quotes,100,0


: 